# 01 - Inlet profile validation

Two parts:
1. **Directional design speed** (NBR 6123 / EN 1991-1-4) from the
   wind-analysis CSVs -- no simulation data needed.
2. **ABL profile validation** per terrain category -- mean velocity + TI vs
   code, spectrum, integral length.

The ABL probe lines are read **directly from the h5 files** (a plain pandas
HDFStore read -- same idea as the old `read_df`), so this notebook does NOT
import `nassu`.

In [ ]:
import json, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from cfdmod import plot_config
from cfdmod.analytical import WindProfile_NBR, WindProfile_EU
from cfdmod.inflow import InflowData, NormalizationParameters
from cfdmod import inflow_report as ir

import logging
logging.getLogger("cfdmod").setLevel(logging.WARNING)
plot_config.apply_style()

project = pathlib.Path("/data/eng/consulting/NNN_CaseName")
case_data = project / "post_processing/pp_config/case_data"
results   = project / "results"
dct_case = json.loads((case_data / "global_data.json").read_text())
batch = dct_case["analysis"]["batch_name"]
H, V0 = float(dct_case["H"]), float(dct_case["V0"])
cats = sorted(k.split("directions_cat")[1] for k in dct_case["analysis"] if k.startswith("directions_cat"))
print(f"H={H} V0={V0} | terrain categories {cats}")

## 1. Directional design speed (NBR + EU)

In [ ]:
wp_nbr = WindProfile_NBR.build(case_data / "wind_analysis_NBR.csv", V0=V0)
wp_eu  = WindProfile_EU.build(case_data / "wind_analysis_EU.csv", Vb=V0)

u_nbr = ir.directional_reference_speed(wp_nbr, height=H, recurrence_period=50, use_kd=True)
u_eu  = ir.directional_reference_speed(wp_eu,  height=H, recurrence_period=50, use_kd=True)
print(f"governing U_H @ {H:g} m: NBR {u_nbr.max():.2f} @ {u_nbr.idxmax():g} deg "
      f"| EU {u_eu.max():.2f} @ {u_eu.idxmax():g} deg")

fig, ax = plot_config.new_axes(xlabel="wind direction [deg]", ylabel="U_H [m/s]",
                               title=f"Directional design speed @ H={H:g} m")
ax.plot(u_nbr.index, u_nbr.to_numpy(), "-o", ms=3, label="NBR 6123")
ax.plot(u_eu.index, u_eu.to_numpy(), "-s", ms=3, label="EN 1991-1-4")
ax.legend()
plt.show()
u_nbr.rename("U_H_NBR").to_frame().join(u_eu.rename("U_H_EU")).round(2)

## 2. ABL profile validation (nassu-free probe read)

`read_line` reads one probe line's velocity h5 straight into a wide frame
(time_step + one column per point), then we reshape to the long form
`InflowData` wants. One representative direction per terrain category.

In [ ]:
def read_line(h5_path):
    """Probe line velocity h5 -> wide DataFrame (time_step + point columns)."""
    parts = []
    with pd.HDFStore(h5_path, mode="r") as store:
        for key in store.keys():
            parts.append(pd.read_hdf(store, key=key))
    return pd.concat(parts).reset_index(drop=True)


def build_inflow(direction, line="line0", component="ux"):
    """Read one ABL probe line into a cfdmod InflowData (no nassu)."""
    folder = (results / batch / f"BaciadaVovo_noBody_{direction}" / "000/probes"
              / "hist_series" / "abl_profile")
    wide = read_line(folder / f"line.{line}.{component}.h5")
    num = [c for c in wide.columns if str(c).isnumeric()]
    long = wide.melt(id_vars=["time_step"], value_vars=num,
                     var_name="point_idx", value_name=component)
    long["point_idx"] = long["point_idx"].astype(int)
    points = pd.read_csv(folder / f"line.{line}.points.csv")
    return InflowData(data=long, points=points), folder

In [ ]:
# representative direction + EU/NBR category labels per terrain category
REP = {"0": ("000.0", "0", "I"), "3": ("022.5", "III", "III")}

measured_u_ref = {}  # per category, handed to the load notebooks via _shared.json
for c in cats:
    rep_dir, cat_eu, cat_nbr = REP.get(c, (dct_case["analysis"][f"directions_cat{c}"][0], "III", "III"))
    inflow, folder = build_inflow(rep_dir)
    if not folder.exists():
        print(f"[cat{c}] no ABL data at {folder}"); continue
    prof = ir.detect_profiles(inflow, min_points=3)[0]
    u_ref = ir.reference_velocity(prof, inflow, H)
    L = ir.integral_length_scale(inflow, prof.nearest_index(H), u_ref)
    print(f"[cat{c}] dir {rep_dir}: u_ref(H)={u_ref:.2f} m/s | L(H)={L:.1f} m")
    measured_u_ref[c] = float(u_ref)
    norm = NormalizationParameters(reference_velocity=max(u_ref, 1e-6), characteristic_length=H)

    fig, _ = ir.plot_profile_vs_code(prof, inflow, H, cat_eu=cat_eu, cat_nbr=cat_nbr)
    plt.show()
    fig = ir.plot_spectrum(prof, inflow, H, norm)
    plt.show()
    L_num = ir.integral_length_scale_profile(inflow, prof)
    fig, _ = ir.plot_integral_length_scale(prof.z, L_num, H)
    plt.show()

## Handoff to the load notebooks

Write the values 02/03 need into a local `_shared.json` in this folder, so
they read them instead of you copy-pasting numbers between notebooks.

In [ ]:
shared = {
    "H": H, "V0": V0,
    "governing_design_U_H": {"NBR": float(u_nbr.max()), "EU": float(u_eu.max())},
    "design_U_H_by_direction_NBR": {f"{float(d):g}": float(v) for d, v in u_nbr.items()},
    "measured_u_ref_by_category": measured_u_ref,
}
out = pathlib.Path.cwd() / "_shared.json"
out.write_text(json.dumps(shared, indent=2))
print("wrote", out)
shared